<a href="https://colab.research.google.com/github/CoderMakar/Plenki/blob/main/Deep4Chem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DATABASE B / Deep4Chem
# ШАГ 1: ЗАГРУЗКА + ПЕРВИЧНАЯ ДИАГНОСТИКА
# ============================================================

import os
import sys
import json
import requests
import subprocess
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. RDKit
# ------------------------------------------------------------

try:
    from rdkit import Chem
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "rdkit"],
        check=True
    )
    from rdkit import Chem


# ============================================================
# 1. CONFIG
# ============================================================

ARTICLE_ID = 12045567
VERSION = 2

BASE_DIR = "/content/Deep4Chem"
os.makedirs(BASE_DIR, exist_ok=True)

API_URL = (
    f"https://api.figshare.com/v2/articles/"
    f"{ARTICLE_ID}/versions/{VERSION}"
)

print("=" * 70)
print("1. FIGSHARE METADATA")
print("=" * 70)

response = requests.get(
    API_URL,
    timeout=60
)

response.raise_for_status()

metadata = response.json()

print("Название:", metadata.get("title"))
print("Версия:", VERSION)

files = metadata.get("files", [])

if not files:
    raise RuntimeError("Figshare не вернул файлов.")


print("\nФайлы:")

for f in files:
    print(
        "-",
        f["name"],
        "|",
        round(f["size"] / 1024 / 1024, 2),
        "MB"
    )


# ============================================================
# 2. DOWNLOAD
# ============================================================

print("\n" + "=" * 70)
print("2. DOWNLOAD")
print("=" * 70)

downloaded = []

for f in files:

    name = f["name"]
    url = f["download_url"]

    path = os.path.join(
        BASE_DIR,
        name
    )

    if not os.path.exists(path):

        print("Скачиваю:", name)

        r = requests.get(
            url,
            timeout=120
        )

        r.raise_for_status()

        with open(path, "wb") as out:
            out.write(r.content)

    else:
        print("Уже существует:", name)

    downloaded.append(path)


# ============================================================
# 3. FIND CSV
# ============================================================

csv_files = [
    p for p in downloaded
    if p.lower().endswith(".csv")
]

if not csv_files:
    raise RuntimeError(
        "CSV в Figshare dataset не найден."
    )

# Если CSV несколько — берём крупнейший
CSV_PATH = max(
    csv_files,
    key=os.path.getsize
)

print(
    "\nОсновной CSV:",
    CSV_PATH
)


# ============================================================
# 4. LOAD CSV
# ============================================================

print("\n" + "=" * 70)
print("3. LOAD DATABASE B")
print("=" * 70)


def load_csv_robust(path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin1",
        "cp1252"
    ]

    last_error = None

    for enc in encodings:

        try:
            return pd.read_csv(
                path,
                encoding=enc
            )

        except Exception as e:
            last_error = e

    raise last_error


df = load_csv_robust(
    CSV_PATH
)

# Чистим пробелы в названиях
df.columns = [
    str(c).strip()
    for c in df.columns
]


print("Размер:", df.shape)

print("\nКолонки:")

for i, col in enumerate(
    df.columns,
    start=1
):
    print(i, repr(col))


print("\nПервые строки:")
display(df.head())


# ============================================================
# 5. EXPECTED COLUMNS
# ============================================================

expected = [
    "Chromophore",
    "Solvent",
    "Absorption max (nm)",
    "log(e/mol-1 dm3 cm-1)"
]

print("\n" + "=" * 70)
print("4. ПРОВЕРКА НУЖНЫХ КОЛОНОК")
print("=" * 70)

for col in expected:

    print(
        col,
        "->",
        "OK"
        if col in df.columns
        else "НЕ НАЙДЕНА"
    )


# ============================================================
# 6. AUTOMATIC COLUMN SEARCH
#
# Если название чуть отличается,
# найдём похожие колонки.
# ============================================================

def find_columns(keyword):

    return [
        c for c in df.columns
        if keyword.lower()
        in c.lower()
    ]


print(
    "\nКолонки с 'abs':",
    find_columns("abs")
)

print(
    "Колонки с 'log':",
    find_columns("log")
)

print(
    "Колонки с 'chrom':",
    find_columns("chrom")
)

print(
    "Колонки с 'solv':",
    find_columns("solv")
)


# ============================================================
# 7. BASIC COUNTS
# ============================================================

print("\n" + "=" * 70)
print("5. BASIC STATISTICS")
print("=" * 70)

if "Chromophore" in df.columns:

    print(
        "Уникальных Chromophore:",
        df["Chromophore"].nunique(
            dropna=True
        )
    )

if "Solvent" in df.columns:

    print(
        "Уникальных Solvent:",
        df["Solvent"].nunique(
            dropna=True
        )
    )


# ============================================================
# 8. TARGET COVERAGE
# ============================================================

ABS_COL = "Absorption max (nm)"
EPS_COL = "log(e/mol-1 dm3 cm-1)"

print("\n" + "=" * 70)
print("6. TARGET COVERAGE")
print("=" * 70)


if ABS_COL in df.columns:

    n_abs = df[
        ABS_COL
    ].notna().sum()

    print(
        "Absorption max есть:",
        n_abs,
        "/",
        len(df),
        f"({n_abs/len(df):.1%})"
    )

    print(
        "\nAbsorption max statistics:"
    )

    print(
        df[ABS_COL]
        .describe()
    )


if EPS_COL in df.columns:

    n_eps = df[
        EPS_COL
    ].notna().sum()

    print(
        "\nlog(epsilon) есть:",
        n_eps,
        "/",
        len(df),
        f"({n_eps/len(df):.1%})"
    )

    print(
        "\nlog(epsilon) statistics:"
    )

    print(
        df[EPS_COL]
        .describe()
    )


if (
    ABS_COL in df.columns
    and
    EPS_COL in df.columns
):

    both_mask = (
        df[ABS_COL].notna()
        &
        df[EPS_COL].notna()
    )

    print(
        "\nОба target одновременно:",
        both_mask.sum(),
        "/",
        len(df),
        f"({both_mask.mean():.1%})"
    )


# ============================================================
# 9. VALIDATE CHROMOPHORE SMILES
# ============================================================

print("\n" + "=" * 70)
print("7. CHROMOPHORE SMILES")
print("=" * 70)


def canonical_smiles(s):

    if pd.isna(s):
        return None

    try:

        mol = Chem.MolFromSmiles(
            str(s)
        )

        if mol is None:
            return None

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except:
        return None


if "Chromophore" in df.columns:

    df["chromophore_canonical"] = (
        df["Chromophore"]
        .apply(
            canonical_smiles
        )
    )

    valid_chrom = (
        df["chromophore_canonical"]
        .notna()
    )

    print(
        "Валидных chromophore SMILES:",
        valid_chrom.sum()
    )

    print(
        "Невалидных:",
        (~valid_chrom).sum()
    )

    print(
        "Уникальных canonical chromophores:",
        df[
            "chromophore_canonical"
        ].nunique(
            dropna=True
        )
    )


# ============================================================
# 10. VALIDATE SOLVENT
# ============================================================

print("\n" + "=" * 70)
print("8. SOLVENT")
print("=" * 70)

if "Solvent" in df.columns:

    df["solvent_canonical"] = (
        df["Solvent"]
        .apply(
            canonical_smiles
        )
    )

    valid_solvent = (
        df["solvent_canonical"]
        .notna()
    )

    print(
        "Валидных solvent SMILES:",
        valid_solvent.sum()
    )

    print(
        "Невалидных / специальных записей:",
        (~valid_solvent).sum()
    )


# ============================================================
# 11. DUPLICATE PAIRS
# ============================================================

print("\n" + "=" * 70)
print("9. CHROMOPHORE + SOLVENT PAIRS")
print("=" * 70)

if (
    "chromophore_canonical"
    in df.columns
    and
    "solvent_canonical"
    in df.columns
):

    valid_pairs = df[
        df["chromophore_canonical"].notna()
        &
        df["solvent_canonical"].notna()
    ]

    duplicates = (
        valid_pairs
        .duplicated(
            subset=[
                "chromophore_canonical",
                "solvent_canonical"
            ],
            keep=False
        )
    )

    print(
        "Валидных пар:",
        len(valid_pairs)
    )

    print(
        "Строк, входящих в повторяющиеся пары:",
        duplicates.sum()
    )

    print(
        "Уникальных пар:",
        valid_pairs[
            [
                "chromophore_canonical",
                "solvent_canonical"
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )


# ============================================================
# 12. TOP SOLVENTS
# ============================================================

print("\n" + "=" * 70)
print("10. TOP SOLVENTS")
print("=" * 70)

if "Solvent" in df.columns:

    top_solvents = (
        df["Solvent"]
        .value_counts(
            dropna=False
        )
        .head(20)
    )

    display(
        top_solvents
        .rename("count")
        .to_frame()
    )


# ============================================================
# 13. TARGETS BY SOLVENT
#
# Это важно: посмотрим,
# хватает ли данных в отдельных средах.
# ============================================================

if (
    "Solvent" in df.columns
    and
    ABS_COL in df.columns
):

    solvent_stats = (
        df.groupby("Solvent")
        .agg(
            n_total=(
                "Chromophore",
                "size"
            ),

            n_abs=(
                ABS_COL,
                lambda x:
                    x.notna().sum()
            ),

            n_logeps=(
                EPS_COL,
                lambda x:
                    x.notna().sum()
            )
        )
        .sort_values(
            "n_abs",
            ascending=False
        )
    )

    print(
        "\nTarget coverage "
        "для самых частых растворителей:"
    )

    display(
        solvent_stats.head(20)
    )


# ============================================================
# 14. SAVE RAW INSPECTED DATA
# ============================================================

OUT_PATH = os.path.join(
    BASE_DIR,
    "Deep4Chem_v2_inspected.csv"
)

df.to_csv(
    OUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("ДИАГНОСТИКА DATABASE B ЗАВЕРШЕНА")
print("=" * 70)

print(
    "Сохранено:",
    OUT_PATH
)

1. FIGSHARE METADATA
Название: DB for chromophore
Версия: 2

Файлы:
- DB for chromophore_Sci_Data_rev02.csv | 3.25 MB

2. DOWNLOAD
Скачиваю: DB for chromophore_Sci_Data_rev02.csv

Основной CSV: /content/Deep4Chem/DB for chromophore_Sci_Data_rev02.csv

3. LOAD DATABASE B
Размер: (20236, 14)

Колонки:
1 'Tag'
2 'Chromophore'
3 'Solvent'
4 'Absorption max (nm)'
5 'Emission max (nm)'
6 'Lifetime (ns)'
7 'Quantum yield'
8 'log(e/mol-1 dm3 cm-1)'
9 'abs FWHM (cm-1)'
10 'emi FWHM (cm-1)'
11 'abs FWHM (nm)'
12 'emi FWHM (nm)'
13 'Molecular weight (g mol-1)'
14 'Reference'

Первые строки:


,Tag,Chromophore,Solvent,Absorption max (nm),Emission max (nm),Lifetime (ns),Quantum yield,log(e/mol-1 dm3 cm-1),abs FWHM (cm-1),emi FWHM (cm-1),abs FWHM (nm),emi FWHM (nm),Molecular weight (g mol-1),Reference
0,1,N#Cc1cc2ccc(O)cc2oc1=O,O,355.0,410.00,2.804262,NaN,NaN,NaN,NaN,NaN,NaN,187.15370,DOI: 10.1021/acs.jpcb.5b09905
1,2,N#Cc1cc2ccc([O-])cc2oc1=O,O,408.0,450.00,3.961965,NaN,NaN,NaN,NaN,NaN,43.0,186.14576,DOI: 10.1021/acs.jpcb.5b09905
2,3,CCCCCCCCCCCC#CC#CCCCCCCCCCN1C(=O)c2ccc3c4ccc5c...,ClC(Cl)Cl,526.0,535.00,3.602954,NaN,NaN,NaN,NaN,NaN,NaN,1061.54348,https://doi.org/10.1002/smll.201901342
3,4,[O-]c1c(-c2nc3ccccc3s2)cc2ccc3cccc4ccc1c2c34,CC#N,514.0,553.72,3.810000,NaN,NaN,NaN,NaN,NaN,67.4,350.42028,https://doi.org/10.1016/j.snb.2018.10.043
4,5,[O-]c1c(-c2nc3ccccc3s2)cc2ccc3cccc4ccc1c2c34,CS(C)=O,524.0,555.00,4.700000,NaN,NaN,NaN,NaN,58.0,50.0,350.42028,https://doi.org/10.1016/j.snb.2018.10.043



4. ПРОВЕРКА НУЖНЫХ КОЛОНОК
Chromophore -> OK
Solvent -> OK
Absorption max (nm) -> OK
log(e/mol-1 dm3 cm-1) -> OK

Колонки с 'abs': ['Absorption max (nm)', 'abs FWHM (cm-1)', 'abs FWHM (nm)']
Колонки с 'log': ['log(e/mol-1 dm3 cm-1)']
Колонки с 'chrom': ['Chromophore']
Колонки с 'solv': ['Solvent']

5. BASIC STATISTICS
Уникальных Chromophore: 6815
Уникальных Solvent: 1336

6. TARGET COVERAGE
Absorption max есть: 17295 / 20236 (85.5%)

Absorption max statistics:
count    17295.000000
mean       427.616868
std        106.102597
min        127.700000
25%        353.000000
50%        397.000000
75%        479.000000
max       1055.018450
Name: Absorption max (nm), dtype: float64

log(epsilon) есть: 8041 / 20236 (39.7%)

log(epsilon) statistics:
count    8041.000000
mean        4.391733
std         0.582147
min         1.079181
25%         4.134878
50%         4.466868
75%         4.731589
max         6.763428
Name: log(e/mol-1 dm3 cm-1), dtype: float64

Оба target одновременно: 8032 / 2023

[11:27:21] SMILES Parse Error: syntax error while parsing: gas
[11:27:21] SMILES Parse Error: check for mistakes around position 1:
[11:27:21] gas
[11:27:21] ^
[11:27:21] SMILES Parse Error: Failed parsing SMILES 'gas' for input: 'gas'
[11:27:22] SMILES Parse Error: syntax error while parsing: gas
[11:27:22] SMILES Parse Error: check for mistakes around position 1:
[11:27:22] gas
[11:27:22] ^
[11:27:22] SMILES Parse Error: Failed parsing SMILES 'gas' for input: 'gas'
[11:27:22] SMILES Parse Error: syntax error while parsing: gas
[11:27:22] SMILES Parse Error: check for mistakes around position 1:
[11:27:22] gas
[11:27:22] ^
[11:27:22] SMILES Parse Error: Failed parsing SMILES 'gas' for input: 'gas'
[11:27:22] SMILES Parse Error: syntax error while parsing: gas
[11:27:22] SMILES Parse Error: check for mistakes around position 1:
[11:27:22] gas
[11:27:22] ^
[11:27:22] SMILES Parse Error: Failed parsing SMILES 'gas' for input: 'gas'
[11:27:22] SMILES Parse Error: syntax error while parsin

Валидных solvent SMILES: 20217
Невалидных / специальных записей: 19

9. CHROMOPHORE + SOLVENT PAIRS
Валидных пар: 20217
Строк, входящих в повторяющиеся пары: 692
Уникальных пар: 19865

10. TOP SOLVENTS


,count
Solvent,
ClCCl,2562
CC#N,2004
Cc1ccccc1,1580
C1CCOC1,1437
ClC(Cl)Cl,1312
CO,1290
CCO,969
CS(C)=O,795
C1CCCCC1,788



Target coverage для самых частых растворителей:


,n_total,n_abs,n_logeps
Solvent,,,
ClCCl,2562,2429,1531
CC#N,2004,1781,893
Cc1ccccc1,1580,1349,574
ClC(Cl)Cl,1312,1249,693
C1CCOC1,1437,1245,668
CO,1290,1183,542
CCO,969,863,440
CS(C)=O,795,703,396
C1CCCCC1,788,658,325



ДИАГНОСТИКА DATABASE B ЗАВЕРШЕНА
Сохранено: /content/Deep4Chem/Deep4Chem_v2_inspected.csv


In [2]:
# ============================================================
# DATABASE B / Deep4Chem
# CLEANING + SPLIT + PROPERTY BASELINE + EXPERT B
#
# Expert B1: Chromophore + Solvent -> lambda_abs_nm
# Expert B2: Chromophore + Solvent -> log_epsilon
#
# Для будущих кандидатов стандартная среда:
# dichloromethane = ClCCl
# ============================================================

import os
import json
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import (
    Descriptors,
    Crippen,
    Lipinski,
    rdMolDescriptors,
    rdFingerprintGenerator
)
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

# Не выводить тысячи RDKit Parse Error
RDLogger.DisableLog("rdApp.error")


# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42

DATA_PATH = (
    "/content/Deep4Chem/"
    "Deep4Chem_v2_inspected.csv"
)

OUTPUT_DIR = (
    "/content/Deep4Chem/Expert_B"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

ABS_COL = "Absorption max (nm)"
EPS_COL = "log(e/mol-1 dm3 cm-1)"

REFERENCE_SOLVENT = "ClCCl"  # dichloromethane

CHROM_FP_SIZE = 1024
SOLVENT_FP_SIZE = 256
FP_RADIUS = 2

AD_REFERENCE_SIZE = 5000

TARGETS = {
    "absorption_nm": ABS_COL,
    "log_epsilon": EPS_COL
}

np.random.seed(SEED)


# ============================================================
# 1. LOAD
# ============================================================

print("=" * 70)
print("1. LOAD DEEP4CHEM")
print("=" * 70)

df = pd.read_csv(
    DATA_PATH
)

print(
    "Исходный размер:",
    df.shape
)


# ============================================================
# 2. CANONICAL SMILES
# ============================================================

def canonicalize(smiles):

    if pd.isna(smiles):
        return None

    try:

        mol = Chem.MolFromSmiles(
            str(smiles)
        )

        if mol is None:
            return None

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except:
        return None


df["chromophore_canonical"] = (
    df["Chromophore"]
    .apply(canonicalize)
)

df["solvent_canonical"] = (
    df["Solvent"]
    .apply(canonicalize)
)


valid = (
    df["chromophore_canonical"].notna()
    &
    df["solvent_canonical"].notna()
)

clean = (
    df.loc[valid]
    .copy()
)

print(
    "После удаления invalid medium:",
    len(clean)
)

print(
    "Удалено:",
    len(df) - len(clean)
)


# ============================================================
# 3. АГРЕГАЦИЯ ПОВТОРНЫХ ИЗМЕРЕНИЙ
#
# Одна строка =
# один chromophore + один solvent.
#
# Если одна и та же пара измерялась несколько раз,
# используем медиану.
# ============================================================

print("\n" + "=" * 70)
print("2. AGGREGATE REPLICATES")
print("=" * 70)


pairs = (
    clean
    .groupby(
        [
            "chromophore_canonical",
            "solvent_canonical"
        ],
        as_index=False
    )
    .agg(
        absorption_nm=(
            ABS_COL,
            "median"
        ),

        log_epsilon=(
            EPS_COL,
            "median"
        ),

        n_abs_measurements=(
            ABS_COL,
            "count"
        ),

        n_logeps_measurements=(
            EPS_COL,
            "count"
        )
    )
)


print(
    "Уникальных chromophore-solvent пар:",
    len(pairs)
)

print(
    "λabs доступно:",
    pairs["absorption_nm"]
    .notna()
    .sum()
)

print(
    "log ε доступно:",
    pairs["log_epsilon"]
    .notna()
    .sum()
)

print(
    "Оба target:",
    (
        pairs["absorption_nm"].notna()
        &
        pairs["log_epsilon"].notna()
    ).sum()
)


# ============================================================
# 4. SCAFFOLD ДЛЯ CHROMOPHORE
# ============================================================

print("\n" + "=" * 70)
print("3. CHROMOPHORE SCAFFOLDS")
print("=" * 70)


def get_scaffold(smiles):

    return (
        MurckoScaffold
        .MurckoScaffoldSmiles(
            smiles=smiles,
            includeChirality=False
        )
    )


chromophores = (
    pairs[
        ["chromophore_canonical"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

chromophores["scaffold"] = (
    chromophores[
        "chromophore_canonical"
    ]
    .apply(get_scaffold)
)


scaffold_counts = (
    chromophores[
        "scaffold"
    ]
    .value_counts()
)


print(
    "Уникальных chromophores:",
    len(chromophores)
)

print(
    "Уникальных scaffolds:",
    len(scaffold_counts)
)

print(
    "Самая большая scaffold-группа:",
    int(scaffold_counts.iloc[0])
)

print(
    "Её доля:",
    round(
        scaffold_counts.iloc[0]
        /
        len(chromophores),
        4
    )
)

print(
    "Acyclic / empty scaffold:",
    (
        chromophores["scaffold"]
        == ""
    ).sum()
)


# ============================================================
# 5. SPLIT
#
# Делим именно CHROMOPHORES,
# а не отдельные измерения.
#
# Поэтому одна молекула не может попасть
# одновременно в train и test
# даже в разных растворителях.
# ============================================================

def scaffold_split(
    chrom_df,
    fractions=(0.8, 0.1, 0.1),
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    scaffold_groups = []

    for scaffold, part in (
        chrom_df
        .groupby("scaffold")
    ):

        indices = (
            part.index
            .to_numpy()
        )

        scaffold_groups.append(
            indices
        )

    # Перемешиваем перед сортировкой,
    # чтобы группы одинакового размера
    # распределялись воспроизводимо
    rng.shuffle(
        scaffold_groups
    )

    scaffold_groups.sort(
        key=len,
        reverse=True
    )

    N = len(chrom_df)

    desired = np.array(
        fractions
    ) * N

    split_indices = [
        [],
        [],
        []
    ]

    counts = np.zeros(
        3,
        dtype=int
    )

    for group in scaffold_groups:

        # Куда сейчас сильнее всего
        # не хватает молекул
        deficits = (
            desired - counts
        )

        possible = [
            i
            for i in range(3)
            if deficits[i] >= len(group)
        ]

        if possible:

            destination = max(
                possible,
                key=lambda i:
                    deficits[i]
            )

        else:

            destination = int(
                np.argmax(
                    deficits
                )
            )

        split_indices[
            destination
        ].extend(
            group.tolist()
        )

        counts[
            destination
        ] += len(group)

    return (
        np.array(
            split_indices[0]
        ),

        np.array(
            split_indices[1]
        ),

        np.array(
            split_indices[2]
        )
    )


largest_fraction = (
    scaffold_counts.iloc[0]
    /
    len(chromophores)
)


USE_SCAFFOLD = (
    len(scaffold_counts) >= 10
    and
    largest_fraction < 0.70
)


# ============================================================
# Fallback:
# если scaffold split вырожден
# ============================================================

if USE_SCAFFOLD:

    train_ch_idx, val_ch_idx, test_ch_idx = (
        scaffold_split(
            chromophores,
            seed=SEED
        )
    )

    SPLIT_TYPE = (
        "Bemis-Murcko scaffold"
    )

else:

    rng = np.random.default_rng(
        SEED
    )

    idx = np.arange(
        len(chromophores)
    )

    rng.shuffle(idx)

    n_train = int(
        0.8 * len(idx)
    )

    n_val = int(
        0.1 * len(idx)
    )

    train_ch_idx = idx[
        :n_train
    ]

    val_ch_idx = idx[
        n_train:
        n_train+n_val
    ]

    test_ch_idx = idx[
        n_train+n_val:
    ]

    SPLIT_TYPE = (
        "chromophore-group random"
    )


chromophores["split"] = ""

chromophores.loc[
    train_ch_idx,
    "split"
] = "train"

chromophores.loc[
    val_ch_idx,
    "split"
] = "validation"

chromophores.loc[
    test_ch_idx,
    "split"
] = "test"


split_map = dict(
    zip(
        chromophores[
            "chromophore_canonical"
        ],

        chromophores[
            "split"
        ]
    )
)


pairs["split"] = (
    pairs[
        "chromophore_canonical"
    ]
    .map(split_map)
)


print(
    "\nSplit type:",
    SPLIT_TYPE
)

print(
    "\nUnique chromophores:"
)

print(
    chromophores[
        "split"
    ].value_counts()
)


if USE_SCAFFOLD:

    tr_scaff = set(
        chromophores.loc[
            train_ch_idx,
            "scaffold"
        ]
    )

    va_scaff = set(
        chromophores.loc[
            val_ch_idx,
            "scaffold"
        ]
    )

    te_scaff = set(
        chromophores.loc[
            test_ch_idx,
            "scaffold"
        ]
    )

    print(
        "\nScaffold overlap train/val:",
        len(
            tr_scaff & va_scaff
        )
    )

    print(
        "Scaffold overlap train/test:",
        len(
            tr_scaff & te_scaff
        )
    )

    print(
        "Scaffold overlap val/test:",
        len(
            va_scaff & te_scaff
        )
    )


# ============================================================
# 6. TARGET DATA COUNTS
# ============================================================

print("\n" + "=" * 70)
print("4. TARGET DATASETS")
print("=" * 70)


for target in TARGETS:

    part = pairs[
        pairs[target].notna()
    ]

    print(
        "\n",
        target
    )

    print(
        part["split"]
        .value_counts()
    )

    print(
        "Total:",
        len(part)
    )


# ============================================================
# 7. SAVE CLEAN DATABASE B
# ============================================================

PAIR_PATH = os.path.join(
    OUTPUT_DIR,
    "Database_B_clean_pairs.csv"
)

pairs.to_csv(
    PAIR_PATH,
    index=False
)


CHROM_SPLIT_PATH = os.path.join(
    OUTPUT_DIR,
    "Database_B_chromophore_split.csv"
)

chromophores.to_csv(
    CHROM_SPLIT_PATH,
    index=False
)


# ============================================================
# 8. DESCRIPTORS
# ============================================================

DESCRIPTOR_NAMES = [
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "FractionCSP3",
    "HeavyAtomCount",
    "AromaticRingCount"
]


def descriptors(mol):

    return np.array(
        [
            Descriptors.MolWt(
                mol
            ),

            Crippen.MolLogP(
                mol
            ),

            rdMolDescriptors.CalcTPSA(
                mol
            ),

            Lipinski.NumHDonors(
                mol
            ),

            Lipinski.NumHAcceptors(
                mol
            ),

            Lipinski.NumRotatableBonds(
                mol
            ),

            rdMolDescriptors.CalcNumRings(
                mol
            ),

            rdMolDescriptors.CalcFractionCSP3(
                mol
            ),

            Descriptors.HeavyAtomCount(
                mol
            ),

            rdMolDescriptors.CalcNumAromaticRings(
                mol
            )
        ],
        dtype=np.float32
    )


# ============================================================
# 9. MORGAN
# ============================================================

chrom_morgan = (
    rdFingerprintGenerator
    .GetMorganGenerator(
        radius=FP_RADIUS,
        fpSize=CHROM_FP_SIZE
    )
)

solvent_morgan = (
    rdFingerprintGenerator
    .GetMorganGenerator(
        radius=FP_RADIUS,
        fpSize=SOLVENT_FP_SIZE
    )
)


def fp_to_array(
    fp,
    size
):

    arr = np.zeros(
        size,
        dtype=np.uint8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    return arr


# ============================================================
# 10. FEATURE GENERATION
#
# Baseline:
# chrom descriptors + solvent descriptors
# = 20 features
#
# Expert B:
# chrom FP 1024
# + solvent FP 256
# + 20 descriptors
# = 1300 features
# ============================================================

print("\n" + "=" * 70)
print("5. FEATURE GENERATION")
print("=" * 70)


N = len(pairs)

X_desc = np.zeros(
    (
        N,
        20
    ),
    dtype=np.float32
)

X_fp = np.zeros(
    (
        N,
        CHROM_FP_SIZE
        +
        SOLVENT_FP_SIZE
    ),
    dtype=np.uint8
)

chrom_fps = []

t0 = time.time()


for i, row in pairs.iterrows():

    chrom = Chem.MolFromSmiles(
        row[
            "chromophore_canonical"
        ]
    )

    solv = Chem.MolFromSmiles(
        row[
            "solvent_canonical"
        ]
    )


    # descriptors
    X_desc[i, :10] = (
        descriptors(
            chrom
        )
    )

    X_desc[i, 10:] = (
        descriptors(
            solv
        )
    )


    # chromophore fingerprint
    cfp = (
        chrom_morgan
        .GetFingerprint(
            chrom
        )
    )

    chrom_fps.append(
        cfp
    )

    X_fp[
        i,
        :CHROM_FP_SIZE
    ] = fp_to_array(
        cfp,
        CHROM_FP_SIZE
    )


    # solvent fingerprint
    sfp = (
        solvent_morgan
        .GetFingerprint(
            solv
        )
    )

    X_fp[
        i,
        CHROM_FP_SIZE:
    ] = fp_to_array(
        sfp,
        SOLVENT_FP_SIZE
    )


X_expert = np.hstack(
    [
        X_fp.astype(
            np.float32
        ),

        X_desc
    ]
)


print(
    "Baseline features:",
    X_desc.shape
)

print(
    "Expert B features:",
    X_expert.shape
)

print(
    "Время:",
    round(
        time.time() - t0,
        1
    ),
    "sec"
)


# ============================================================
# 11. METRICS
# ============================================================

def regression_metrics(
    true,
    pred
):

    return {
        "MAE":
            float(
                mean_absolute_error(
                    true,
                    pred
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    mean_squared_error(
                        true,
                        pred
                    )
                )
            ),

        "R2":
            float(
                r2_score(
                    true,
                    pred
                )
            )
    }


# ============================================================
# 12. PROPERTY BASELINE
#
# Это НЕ B0 генератор!
# ============================================================

def make_baseline():

    return RandomForestRegressor(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )


# ============================================================
# 13. EXPERT B
# ============================================================

def make_expert():

    return ExtraTreesRegressor(
        n_estimators=250,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1
    )


# ============================================================
# 14. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("6. TRAIN PROPERTY BASELINE + EXPERT B")
print("=" * 70)


results = []

baselines = {}
experts = {}


for target in TARGETS:

    print(
        "\nTARGET:",
        target
    )

    valid_target = (
        pairs[target]
        .notna()
        .values
    )

    train_idx = np.where(
        valid_target
        &
        (
            pairs["split"]
            .values
            == "train"
        )
    )[0]

    val_idx = np.where(
        valid_target
        &
        (
            pairs["split"]
            .values
            == "validation"
        )
    )[0]

    test_idx = np.where(
        valid_target
        &
        (
            pairs["split"]
            .values
            == "test"
        )
    )[0]


    y = (
        pairs[target]
        .values
        .astype(
            np.float32
        )
    )


    print(
        "Train:",
        len(train_idx)
    )

    print(
        "Validation:",
        len(val_idx)
    )

    print(
        "Test:",
        len(test_idx)
    )


    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline = (
        make_baseline()
    )

    baseline.fit(
        X_desc[train_idx],
        y[train_idx]
    )

    val_pred = (
        baseline.predict(
            X_desc[val_idx]
        )
    )

    m = regression_metrics(
        y[val_idx],
        val_pred
    )

    print(
        "\nProperty baseline / validation:"
    )

    print(m)

    results.append(
        {
            "target":
                target,

            "model":
                "property_baseline",

            "split":
                "validation",

            **m
        }
    )


    # --------------------------------------------------------
    # Expert B
    # --------------------------------------------------------

    expert = (
        make_expert()
    )

    expert.fit(
        X_expert[train_idx],
        y[train_idx]
    )

    val_pred = (
        expert.predict(
            X_expert[val_idx]
        )
    )

    m = regression_metrics(
        y[val_idx],
        val_pred
    )

    print(
        "\nExpert B / validation:"
    )

    print(m)

    results.append(
        {
            "target":
                target,

            "model":
                "expert_B",

            "split":
                "validation",

            **m
        }
    )


    # --------------------------------------------------------
    # Internal test
    # --------------------------------------------------------

    baseline_test = (
        baseline.predict(
            X_desc[test_idx]
        )
    )

    expert_test = (
        expert.predict(
            X_expert[test_idx]
        )
    )


    bm = regression_metrics(
        y[test_idx],
        baseline_test
    )

    em = regression_metrics(
        y[test_idx],
        expert_test
    )


    print(
        "\nProperty baseline / TEST:"
    )

    print(bm)

    print(
        "Expert B / TEST:"
    )

    print(em)


    results.append(
        {
            "target":
                target,

            "model":
                "property_baseline",

            "split":
                "test",

            **bm
        }
    )

    results.append(
        {
            "target":
                target,

            "model":
                "expert_B",

            "split":
                "test",

            **em
        }
    )


    baselines[target] = (
        baseline
    )

    experts[target] = (
        expert
    )


# ============================================================
# 15. RESULTS
# ============================================================

metrics_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("7. METRICS")
print("=" * 70)

display(
    metrics_df.sort_values(
        [
            "target",
            "split",
            "model"
        ]
    )
)


# ============================================================
# 16. TREE UNCERTAINTY
# ============================================================

def tree_uncertainty(
    model,
    X
):

    predictions = np.vstack(
        [
            tree.predict(X)
            for tree
            in model.estimators_
        ]
    )

    return (
        predictions.mean(
            axis=0
        ),

        predictions.std(
            axis=0
        )
    )


# ============================================================
# 17. APPLICABILITY DOMAIN B
#
# AD рассчитываем только по CHROMOPHORE.
#
# Для будущих кандидатов растворитель
# будет фиксирован = ClCCl.
# ============================================================

print("\n" + "=" * 70)
print("8. APPLICABILITY DOMAIN B")
print("=" * 70)


ad_data = {}


for target in TARGETS:

    valid_target = (
        pairs[target]
        .notna()
    )


    # -----------------------------------------
    # Unique train chromophores for this target
    # -----------------------------------------

    train_chrom = (
        pairs.loc[
            valid_target
            &
            (
                pairs["split"]
                == "train"
            ),
            "chromophore_canonical"
        ]
        .drop_duplicates()
        .tolist()
    )


    val_chrom = (
        pairs.loc[
            valid_target
            &
            (
                pairs["split"]
                == "validation"
            ),
            "chromophore_canonical"
        ]
        .drop_duplicates()
        .tolist()
    )


    rng = np.random.default_rng(
        SEED
    )

    if len(train_chrom) > AD_REFERENCE_SIZE:

        train_chrom = list(
            rng.choice(
                train_chrom,
                AD_REFERENCE_SIZE,
                replace=False
            )
        )


    reference_fps = []

    for smi in train_chrom:

        mol = Chem.MolFromSmiles(
            smi
        )

        reference_fps.append(
            chrom_morgan
            .GetFingerprint(
                mol
            )
        )


    def max_similarity_to_B(
        smiles
    ):

        mol = Chem.MolFromSmiles(
            smiles
        )

        fp = (
            chrom_morgan
            .GetFingerprint(
                mol
            )
        )

        sims = (
            DataStructs
            .BulkTanimotoSimilarity(
                fp,
                reference_fps
            )
        )

        return float(
            max(sims)
        )


    val_sims = [
        max_similarity_to_B(
            smi
        )
        for smi in val_chrom
    ]


    threshold = float(
        np.quantile(
            val_sims,
            0.05
        )
    )


    ad_data[target] = {
        "threshold":
            threshold,

        "reference_smiles":
            train_chrom,

        "reference_fps":
            reference_fps
    }


    print(
        "\n",
        target
    )

    print(
        "AD reference molecules:",
        len(train_chrom)
    )

    print(
        "AD threshold:",
        round(
            threshold,
            4
        )
    )

    print(
        "Median validation similarity:",
        round(
            float(
                np.median(
                    val_sims
                )
            ),
            4
        )
    )


# ============================================================
# 18. SAVE MODELS
# ============================================================

print("\n" + "=" * 70)
print("9. SAVE MODELS")
print("=" * 70)


for target in TARGETS:

    joblib.dump(
        baselines[target],
        os.path.join(
            OUTPUT_DIR,
            f"property_baseline_B_{target}.joblib"
        )
    )

    joblib.dump(
        experts[target],
        os.path.join(
            OUTPUT_DIR,
            f"expert_B_{target}.joblib"
        )
    )


metrics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Expert_B_metrics.csv"
    ),
    index=False
)


# ============================================================
# 19. SAVE AD REFERENCES
# ============================================================

for target in TARGETS:

    pd.DataFrame(
        {
            "chromophore_smiles":
                ad_data[
                    target
                ][
                    "reference_smiles"
                ]
        }
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"AD_reference_B_{target}.csv"
        ),
        index=False
    )


# ============================================================
# 20. PREDICTION FEATURE FUNCTION
# ============================================================

reference_solvent_canonical = (
    canonicalize(
        REFERENCE_SOLVENT
    )
)

assert (
    reference_solvent_canonical
    is not None
)


def make_features_for_candidates(
    smiles_list,
    solvent_smiles=REFERENCE_SOLVENT
):

    solvent_canonical = (
        canonicalize(
            solvent_smiles
        )
    )

    if solvent_canonical is None:

        raise ValueError(
            "Invalid solvent SMILES"
        )


    solvent_mol = (
        Chem.MolFromSmiles(
            solvent_canonical
        )
    )

    solvent_desc = (
        descriptors(
            solvent_mol
        )
    )

    solvent_fp = (
        solvent_morgan
        .GetFingerprint(
            solvent_mol
        )
    )

    solvent_fp_array = (
        fp_to_array(
            solvent_fp,
            SOLVENT_FP_SIZE
        )
    )


    valid_smiles = []
    valid_indices = []

    desc_rows = []
    expert_rows = []

    candidate_fps = []


    for i, smi in enumerate(
        smiles_list
    ):

        canon = canonicalize(
            smi
        )

        if canon is None:
            continue


        mol = Chem.MolFromSmiles(
            canon
        )

        cdesc = (
            descriptors(
                mol
            )
        )

        cfp = (
            chrom_morgan
            .GetFingerprint(
                mol
            )
        )

        cfp_array = (
            fp_to_array(
                cfp,
                CHROM_FP_SIZE
            )
        )


        desc_vector = np.concatenate(
            [
                cdesc,
                solvent_desc
            ]
        )


        expert_vector = np.concatenate(
            [
                cfp_array.astype(
                    np.float32
                ),

                solvent_fp_array.astype(
                    np.float32
                ),

                desc_vector
            ]
        )


        valid_smiles.append(
            canon
        )

        valid_indices.append(
            i
        )

        desc_rows.append(
            desc_vector
        )

        expert_rows.append(
            expert_vector
        )

        candidate_fps.append(
            cfp
        )


    return (
        valid_indices,
        valid_smiles,
        np.asarray(
            desc_rows,
            dtype=np.float32
        ),
        np.asarray(
            expert_rows,
            dtype=np.float32
        ),
        candidate_fps
    )


# ============================================================
# 21. FINAL EXPERT B FUNCTION
# ============================================================

def predict_expert_B(
    smiles_list,
    solvent_smiles=REFERENCE_SOLVENT
):

    result = pd.DataFrame(
        {
            "smiles":
                smiles_list,

            "valid":
                False
        }
    )


    (
        valid_indices,
        valid_smiles,
        Xd,
        Xe,
        candidate_fps
    ) = make_features_for_candidates(
        smiles_list,
        solvent_smiles
    )


    if len(valid_indices) == 0:
        return result


    result.loc[
        valid_indices,
        "valid"
    ] = True


    result.loc[
        valid_indices,
        "canonical_smiles"
    ] = valid_smiles


    result.loc[
        valid_indices,
        "evaluation_solvent"
    ] = canonicalize(
        solvent_smiles
    )


    # --------------------------------------------------------
    # Predictions + uncertainty
    # --------------------------------------------------------

    for target in TARGETS:

        mean, uncertainty = (
            tree_uncertainty(
                experts[target],
                Xe
            )
        )

        result.loc[
            valid_indices,
            f"pred_{target}"
        ] = mean

        result.loc[
            valid_indices,
            f"uncertainty_{target}"
        ] = uncertainty


        reference_fps = (
            ad_data[target][
                "reference_fps"
            ]
        )

        threshold = (
            ad_data[target][
                "threshold"
            ]
        )


        similarities = []

        for fp in candidate_fps:

            sims = (
                DataStructs
                .BulkTanimotoSimilarity(
                    fp,
                    reference_fps
                )
            )

            similarities.append(
                max(sims)
            )


        result.loc[
            valid_indices,
            f"max_train_tanimoto_B_{target}"
        ] = similarities


        result.loc[
            valid_indices,
            f"in_domain_B_{target}"
        ] = (
            np.asarray(
                similarities
            )
            >= threshold
        )


    # Общий domain B:
    # кандидат должен быть внутри домена
    # ОБОИХ UV models

    result.loc[
        valid_indices,
        "in_domain_B"
    ] = (
        result.loc[
            valid_indices,
            "in_domain_B_absorption_nm"
        ].astype(bool)
        &
        result.loc[
            valid_indices,
            "in_domain_B_log_epsilon"
        ].astype(bool)
    )


    return result


# ============================================================
# 22. QUICK TEST
# ============================================================

print("\n" + "=" * 70)
print("10. QUICK EXPERT B TEST")
print("=" * 70)


example_smiles = (
    pairs[
        pairs["split"]
        == "test"
    ][
        "chromophore_canonical"
    ]
    .drop_duplicates()
    .head(5)
    .tolist()
)


display(
    predict_expert_B(
        example_smiles
    )
)


# ============================================================
# 23. CONFIG
# ============================================================

config = {

    "seed":
        SEED,

    "source":
        "Deep4Chem v2",

    "split_type":
        SPLIT_TYPE,

    "targets": [
        "absorption_nm",
        "log_epsilon"
    ],

    "reference_solvent":
        reference_solvent_canonical,

    "reference_solvent_name":
        "dichloromethane",

    "chromophore_fingerprint": {
        "type":
            "Morgan",

        "radius":
            FP_RADIUS,

        "bits":
            CHROM_FP_SIZE
    },

    "solvent_fingerprint": {
        "type":
            "Morgan",

        "radius":
            FP_RADIUS,

        "bits":
            SOLVENT_FP_SIZE
    },

    "property_baseline":
        (
            "RandomForest on "
            "chromophore + solvent descriptors"
        ),

    "expert_B":
        (
            "ExtraTrees on chromophore + "
            "solvent fingerprints and descriptors"
        ),

    "uncertainty":
        (
            "standard deviation between "
            "ExtraTrees estimators"
        ),

    "AD_absorption_nm":
        ad_data[
            "absorption_nm"
        ][
            "threshold"
        ],

    "AD_log_epsilon":
        ad_data[
            "log_epsilon"
        ][
            "threshold"
        ]
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "Expert_B_config.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 24. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("DATABASE B / EXPERT B ГОТОВ")
print("=" * 70)

print(
    "\nTarget B1: λabs,max"
)

print(
    "Target B2: log10(epsilon)"
)

print(
    "\nReference solvent:",
    reference_solvent_canonical,
    "(dichloromethane)"
)

print(
    "\nSplit:",
    SPLIT_TYPE
)

print(
    "\nРезультаты:",
    OUTPUT_DIR
)

1. LOAD DEEP4CHEM
Исходный размер: (20236, 16)
После удаления invalid medium: 20217
Удалено: 19

2. AGGREGATE REPLICATES
Уникальных chromophore-solvent пар: 19865
λabs доступно: 16968
log ε доступно: 7909
Оба target: 7900

3. CHROMOPHORE SCAFFOLDS
Уникальных chromophores: 6783
Уникальных scaffolds: 3494
Самая большая scaffold-группа: 101
Её доля: 0.0149
Acyclic / empty scaffold: 23

Split type: Bemis-Murcko scaffold

Unique chromophores:
split
train         5427
test           678
validation     678
Name: count, dtype: int64

Scaffold overlap train/val: 0
Scaffold overlap train/test: 0
Scaffold overlap val/test: 0

4. TARGET DATASETS

 absorption_nm
split
train         13944
validation     1601
test           1423
Name: count, dtype: int64
Total: 16968

 log_epsilon
split
train         6507
validation     768
test           634
Name: count, dtype: int64
Total: 7909

5. FEATURE GENERATION
Baseline features: (19865, 20)
Expert B features: (19865, 1300)
Время: 43.2 sec

6. TRAIN PROPERTY 

,target,model,split,MAE,RMSE,R2
3,absorption_nm,expert_B,test,29.823124,44.111582,0.800251
2,absorption_nm,property_baseline,test,52.931903,71.635367,0.473213
1,absorption_nm,expert_B,validation,36.186625,51.811222,0.738258
0,absorption_nm,property_baseline,validation,56.058303,77.460235,0.414964
7,log_epsilon,expert_B,test,0.212002,0.298186,0.699619
6,log_epsilon,property_baseline,test,0.328341,0.488598,0.193505
5,log_epsilon,expert_B,validation,0.221821,0.303313,0.531024
4,log_epsilon,property_baseline,validation,0.304962,0.424011,0.083520



8. APPLICABILITY DOMAIN B

 absorption_nm
AD reference molecules: 5000
AD threshold: 0.3708
Median validation similarity: 0.6667

 log_epsilon
AD reference molecules: 3097
AD threshold: 0.385
Median validation similarity: 0.6645

9. SAVE MODELS

10. QUICK EXPERT B TEST


,smiles,valid,canonical_smiles,evaluation_solvent,pred_absorption_nm,uncertainty_absorption_nm,max_train_tanimoto_B_absorption_nm,in_domain_B_absorption_nm,pred_log_epsilon,uncertainty_log_epsilon,max_train_tanimoto_B_log_epsilon,in_domain_B_log_epsilon,in_domain_B
0,Brc1cnc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)s1,True,Brc1cnc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)s1,ClCCl,383.806029,41.825224,0.705882,True,4.414791,0.264309,0.705882,True,True
1,Brc1cnc(-c2ccncc2)s1,True,Brc1cnc(-c2ccncc2)s1,ClCCl,343.468734,53.332239,0.435897,True,4.055493,0.407434,0.435897,True,True
2,C#Cc1ccc(-c2nc(-c3ccc(C#C)s3)nc(-c3ccc(C#C)s3)...,True,C#Cc1ccc(-c2nc(-c3ccc(C#C)s3)nc(-c3ccc(C#C)s3)...,ClCCl,364.486586,44.361597,0.333333,False,4.332057,0.405349,0.282051,False,False
3,C(#Cc1cccc2ccccc12)c1ncc(-c2ccccc2)o1,True,C(#Cc1cccc2ccccc12)c1ncc(-c2ccccc2)o1,ClCCl,357.441000,50.221597,0.547619,True,4.314756,0.264443,0.547619,True,True
4,C(#Cc1ccccc1)C(C#Cc1ccccc1)=Cc1c2ccccc2c(C=C(C...,True,C(#Cc1ccccc1)C(C#Cc1ccccc1)=Cc1c2ccccc2c(C=C(C...,ClCCl,401.606418,53.479267,0.666667,True,4.588224,0.339005,0.666667,True,True



DATABASE B / EXPERT B ГОТОВ

Target B1: λabs,max
Target B2: log10(epsilon)

Reference solvent: ClCCl (dichloromethane)

Split: Bemis-Murcko scaffold

Результаты: /content/Deep4Chem/Expert_B
